# MEDIC SPI training campaign
Image-only four-class pseudo-label agreement. Separate post-text targets where available. Four frozen encoders, nine vector-head families, attention heads and mean-pool controls. Kaggle managed Python; setup prints exact environment and source SHA. Test locked until model selection freezes. Training outputs remain research candidates; no automatic promotion. Artifacts/logs are incremental. No full dataset archives.

In [ ]:
import os,sys,time,json,platform,subprocess,hashlib,base64,io,zipfile,traceback
from pathlib import Path
print('SETUP START',sys.executable,sys.version,flush=True)
PROJECT=Path('/kaggle/working/project');PROJECT.mkdir(exist_ok=True)
RUN=Path('/kaggle/working/experiments/spi_classifier_repair/runs')/('medic_spi_'+time.strftime('%Y%m%dT%H%M%SZ',time.gmtime()));RUN.mkdir(parents=True)
os.environ['YOLO_CONFIG_DIR']=str(RUN/'yolo_config');os.environ['OMP_NUM_THREADS']='1';os.environ['MKL_NUM_THREADS']='1'
class Tee:
 def __init__(self,stream,path): self.stream=stream;self.file=open(path,'a',buffering=1)
 def write(self,s): self.stream.write(s);self.file.write(s);return len(s)
 def flush(self): self.stream.flush();self.file.flush()
sys.stdout=Tee(sys.stdout,RUN/'stdout.log');sys.stderr=Tee(sys.stderr,RUN/'stderr.log')
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='7d894a0654fd89cf22af08a0c90ae89aa2c992993b857ad0549516fb5cc3fae0'
with zipfile.ZipFile(io.BytesIO(blob)) as z:z.extractall(PROJECT)
sys.path.insert(0,str(PROJECT));os.environ['PYTHONPATH']=str(PROJECT)
print('SOURCE VERIFIED',hashlib.sha256(blob).hexdigest(),flush=True)
import torch
print('HARDWARE',torch.__version__,torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
assert torch.cuda.device_count()==2,'two GPUs required'
commands=[[sys.executable,'-m','pip','install','transformers==5.5.0','ultralytics==8.3.237','ftfy','regex','requests','nbformat','bitsandbytes','accelerate','scikit-learn==1.7.2','joblib','threadpoolctl','psutil','matplotlib'],[sys.executable,'-m','pip','install','--no-deps','--no-build-isolation','git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33']]
for i,command in enumerate(commands):
 print('DEPENDENCIES START',i,flush=True)
 with (RUN/f'install_{i}.log').open('w') as log:subprocess.run(command,stdout=log,stderr=subprocess.STDOUT,check=True,timeout=300)
 print('DEPENDENCIES END',i,flush=True)
import importlib.metadata as md
(RUN/'environment.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,'packages':{k:md.version(k) for k in ['transformers','ultralytics','numpy','huggingface-hub']},'source_sha256':hashlib.sha256(blob).hexdigest()},indent=2))
print('SETUP END',RUN,flush=True)


In [ ]:
print('EXACT MEDIC COHORT LOAD START',flush=True)
from parameters.artifacts import write_json
found=list(Path('/kaggle/input').glob('**/grid/data/records.json'))
if len(found)!=1:raise RuntimeError('Attach exactly one verified v18 MEDIC grid output; no automatic reacquisition or resampling')
rows=json.loads(found[0].read_text())
for row in rows:
 matches=list(found[0].parent.glob('images/'+row['image_sha256']+'.img'))
 if len(matches)!=1:raise RuntimeError('exact upstream image missing')
 row['image_path']=str(matches[0]);row['prior_detector_study_exposure']=True
if len(rows)!=300 or any(r['source']!='medic' for r in rows):raise RuntimeError('MEDIC300 cohort mismatch')
write_json(RUN/'data/records.json',rows)
print('EXACT MEDIC COHORT LOAD END',len(rows),flush=True)


In [ ]:
print('SPI CAMPAIGN START',flush=True)
from experiments.spi_classifier_repair.medic_campaign import run
try:
 run(RUN/'data/records.json',PROJECT/'parameters/results/spi_inputs_v2_20261027b/panel.json',PROJECT/'parameters/config/final_labeling_teacher.json',RUN/'campaign')
except BaseException:
 (RUN/'failure.txt').write_text(traceback.format_exc());raise
finally:
 print('SPI CAMPAIGN END; inspect receipts, not notebook completion alone',RUN,flush=True)
